# DengueCast v4 — Resultados e Visualizações

**Dataset:** [`pedrolucassantanaf/dengue-tcc2-data`](https://huggingface.co/datasets/pedrolucassantanaf/dengue-tcc2-data)  
**Dashboard:** [modelo-previsao-ffqc3ugltpbxxeaa4mygvw.streamlit.app](https://modelo-previsao-ffqc3ugltpbxxeaa4mygvw.streamlit.app)  
**Pipeline:** v4 — 137 mesorregiões, XGBoost

---

Visualizações dos resultados do modelo B_inmet_enriquecido: scatter plots, séries temporais, métricas por UF, SHAP, walk-forward e matriz de confusão.

In [ ]:
%%capture
!pip install huggingface_hub xgboost plotly kaleido scikit-learn

In [ ]:
import os, json
import numpy as np
import pandas as pd
import xgboost as xgb
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import r2_score, confusion_matrix
from huggingface_hub import hf_hub_download

IS_KAGGLE = "KAGGLE_DATA_DIR" in os.environ
if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
else:
    HF_TOKEN = os.environ.get("HF_TOKEN")

REPO = os.environ.get("HF_REPO_ID", "pedrolucassantanaf/dengue-tcc2-data")

def hf_download(path):
    return hf_hub_download(REPO, path, repo_type="dataset", token=HF_TOKEN)

COLORS = {"primary": "#5046e5", "green": "#22c55e", "amber": "#f59e0b",
          "orange": "#f97316", "red": "#ef4444", "gray": "#94a3b8"}

print(f"Ambiente: {'Kaggle' if IS_KAGGLE else 'Local'}")

## 1. Carregar dados e modelos

In [ ]:
test = pd.read_parquet(hf_download("data/model_ready_v4/splits_b/test.parquet"))
train = pd.read_parquet(hf_download("data/model_ready_v4/splits_b/train.parquet"))
schema = pd.read_csv(hf_download("data/model_ready_v4/feature_schema_v4.csv"))
features = schema["feature"].tolist()

TARGET = "notificacoes_t4"

# Carregar modelo pré-treinado do HF (ou local se treinou no notebook 2)
local_model = "models_trained/model_B_inmet_enriquecido.ubj"
if os.path.exists(local_model):
    print("Usando modelo treinado localmente (notebook 2)")
    model_path = local_model
else:
    print("Baixando modelo pré-treinado do HuggingFace")
    model_path = hf_download("models/regression_v4/model_B_inmet_enriquecido.ubj")

reg_model = xgb.XGBRegressor()
reg_model.load_model(model_path)

# Modelo de classificação
local_cls = "models_trained/model_B_inmet_enriquecido_cls.ubj"
if os.path.exists(local_cls):
    cls_path = local_cls
else:
    cls_path = hf_download("models/classification_v4/model_B_inmet_enriquecido_cls.ubj")
cls_model = xgb.XGBClassifier()
cls_model.load_model(cls_path)

# Alinhar features na ordem que o modelo espera (pode diferir do schema CSV)
model_features = reg_model.get_booster().feature_names
if model_features and set(model_features) == set(features):
    features = model_features

# Previsões
y_test_log = np.log1p(test[TARGET])
y_pred_log = reg_model.predict(test[features])
y_test_orig = test[TARGET].values
y_pred_orig = np.expm1(y_pred_log)

r2_log = r2_score(y_test_log, y_pred_log)
print(f"R²_log no teste: {r2_log:.4f}")

## 2. Scatter: previsão vs real (escala log)

In [ ]:
risk_labels = {0: "Baixo", 1: "Moderado", 2: "Alto", 3: "Surto"}
risk_colors = {"Baixo": COLORS["green"], "Moderado": COLORS["amber"],
               "Alto": COLORS["orange"], "Surto": COLORS["red"]}

scatter_df = pd.DataFrame({
    "Real (log1p)": y_test_log,
    "Previsto (log1p)": y_pred_log,
    "Risco": test["risco_surto_t4"].map(risk_labels),
})

fig = px.scatter(scatter_df, x="Real (log1p)", y="Previsto (log1p)", color="Risco",
                 color_discrete_map=risk_colors, opacity=0.4,
                 title=f"Previsão vs Real — B_inmet_enriquecido (R²_log = {r2_log:.4f})")

r = [y_test_log.min(), y_test_log.max()]
fig.add_trace(go.Scatter(x=r, y=r, mode="lines", line=dict(dash="dash", color="black", width=1),
                         showlegend=False, name="Ideal"))
fig.update_layout(font=dict(family="DM Sans"), template="plotly_white",
                  width=700, height=600)
fig.show()

## 3. Série temporal nacional: real vs prevista

In [ ]:
test["pred_orig"] = y_pred_orig

ts = test.groupby(["ano", "semana_epidemiologica"]).agg(
    real=(TARGET, "sum"),
    previsto=("pred_orig", "sum")
).reset_index()
ts["periodo"] = ts["ano"].astype(str) + "-S" + ts["semana_epidemiologica"].astype(str).str.zfill(2)

fig = go.Figure()
fig.add_trace(go.Scatter(x=ts["periodo"], y=ts["real"], name="Real",
                         line=dict(color=COLORS["primary"], width=2)))
fig.add_trace(go.Scatter(x=ts["periodo"], y=ts["previsto"], name="Previsto",
                         line=dict(color=COLORS["red"], width=2, dash="dash")))
fig.update_layout(title="Série temporal nacional: real vs previsto (soma semanal)",
                  xaxis_title="Período", yaxis_title="Total de notificações",
                  font=dict(family="DM Sans"), template="plotly_white")
fig.show()

## 4. R² por ano

In [ ]:
all_data = pd.concat([train, test], ignore_index=True)
all_data["pred_log"] = np.nan
all_data.loc[all_data.index[:len(train)], "pred_log"] = reg_model.predict(train[features])
all_data.loc[all_data.index[len(train):], "pred_log"] = y_pred_log

r2_by_year = []
for ano, g in all_data.dropna(subset=["pred_log"]).groupby("ano"):
    r2 = r2_score(np.log1p(g[TARGET]), g["pred_log"])
    split = "treino" if ano <= 2022 else ("validação" if ano == 2023 else "teste")
    r2_by_year.append({"Ano": ano, "R²_log": r2, "Split": split})

r2_df = pd.DataFrame(r2_by_year)
fig = px.bar(r2_df, x="Ano", y="R²_log", color="Split",
             color_discrete_map={"treino": COLORS["primary"], "validação": COLORS["amber"], "teste": COLORS["red"]},
             title="R² (log) por ano")
fig.update_layout(font=dict(family="DM Sans"), template="plotly_white",
                  yaxis=dict(range=[0, 1]))
fig.show()

## 5. Métricas por UF

In [ ]:
metrics_uf = pd.read_csv(hf_download("models/regression_v4/metrics_por_uf_v4.csv"))

# Se o CSV tem coluna R2_log, use; senão calcule
r2_col = [c for c in metrics_uf.columns if 'r2' in c.lower() or 'R2' in c]
print(f"Colunas disponíveis: {metrics_uf.columns.tolist()}")
print(f"\nTop 10 UFs:")
print(metrics_uf.sort_values(metrics_uf.columns[1], ascending=False).head(10).to_string(index=False))

## 6. SHAP: top-20 features

In [ ]:
shap_df = pd.read_csv(hf_download("models/regression_v4/shap_importance_v4.csv"))

print(f"Colunas SHAP: {shap_df.columns.tolist()}")

# Pegar top 20
importance_col = [c for c in shap_df.columns if 'importance' in c.lower() or 'shap' in c.lower() or 'mean' in c.lower()]
if importance_col:
    val_col = importance_col[0]
else:
    val_col = shap_df.columns[1]

feat_col = shap_df.columns[0]
top20 = shap_df.nlargest(20, val_col)

# Adicionar origem (SINAN/INMET)
schema_map = dict(zip(schema["feature"], schema["category"]))
top20 = top20.copy()
top20["origem"] = top20[feat_col].map(lambda f: "SINAN" if schema_map.get(f) == "sinan" else "INMET")

fig = px.bar(top20, x=val_col, y=feat_col, color="origem",
             color_discrete_map={"SINAN": COLORS["primary"], "INMET": COLORS["green"]},
             orientation="h", title="Top-20 features por importância SHAP")
fig.update_layout(yaxis=dict(autorange="reversed"),
                  font=dict(family="DM Sans"), template="plotly_white",
                  height=600)
fig.show()

## 7. Walk-forward por ano

In [ ]:
wf_report = json.loads(open(hf_download("data/model_ready_v4/47_walkforward_v4_report.json")).read())

# Extrair walk-forward results
if isinstance(wf_report, dict):
    # Tentar encontrar os dados de walk-forward
    wf_data = []
    for key, val in wf_report.items():
        if isinstance(val, list):
            for item in val:
                if isinstance(item, dict) and any(k in item for k in ["fold", "ano", "year", "R2_log", "r2_log"]):
                    wf_data.append(item)
            if wf_data:
                break
        elif isinstance(val, dict):
            for k2, v2 in val.items():
                if isinstance(v2, list):
                    for item in v2:
                        if isinstance(item, dict):
                            wf_data.append(item)
                    if wf_data:
                        break

    if wf_data:
        wf_df = pd.DataFrame(wf_data)
        print(f"Walk-forward: {len(wf_df)} folds")
        print(wf_df.head())
    else:
        print("Estrutura do walk-forward report:")
        for k, v in wf_report.items():
            print(f"  {k}: {type(v).__name__} ({len(v) if hasattr(v, '__len__') else 'scalar'})")

## 8. Comparativo de variantes (A vs B vs C)

In [ ]:
reg_report = json.loads(open(hf_download("data/model_ready_v4/46_regression_v4_report.json")).read())

rows = []
for model_key, metrics in reg_report["modelos"].items():
    variant = model_key.split("_")[0]
    arm = "_".join(model_key.split("_")[1:])
    rows.append({
        "Variante": variant,
        "Braço": arm,
        "R²_log": metrics["R2_log"],
        "CI_lo": metrics.get("R2_log_CI95_lo", metrics["R2_log"] - 0.005),
        "CI_hi": metrics.get("R2_log_CI95_hi", metrics["R2_log"] + 0.005),
    })

comp_df = pd.DataFrame(rows)
comp_df["label"] = comp_df["Variante"] + "_" + comp_df["Braço"]

variant_colors = {"A": COLORS["amber"], "B": COLORS["primary"], "C": COLORS["green"]}

fig = go.Figure()
for variant in ["A", "B", "C"]:
    sub = comp_df[comp_df["Variante"] == variant]
    fig.add_trace(go.Bar(
        x=sub["Braço"], y=sub["R²_log"], name=f"Variante {variant}",
        marker_color=variant_colors[variant],
        error_y=dict(type="data",
                     symmetric=False,
                     array=(sub["CI_hi"] - sub["R²_log"]).tolist(),
                     arrayminus=(sub["R²_log"] - sub["CI_lo"]).tolist()),
    ))

fig.update_layout(title="Comparativo A vs B vs C (R²_log com IC 95%)",
                  barmode="group", yaxis=dict(range=[0.85, 0.90]),
                  font=dict(family="DM Sans"), template="plotly_white")
fig.show()

## 9. Matriz de confusão: classificação de risco

In [ ]:
y_test_cls = test["risco_surto_t4"]
y_pred_cls = cls_model.predict(test[features])

cm = confusion_matrix(y_test_cls, y_pred_cls, labels=[0, 1, 2, 3])
labels = ["Baixo", "Moderado", "Alto", "Surto"]

# Normalizar por linha (recall)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

text = [[f"{cm[i][j]}\n({cm_pct[i][j]:.0f}%)" for j in range(4)] for i in range(4)]

fig = go.Figure(data=go.Heatmap(
    z=cm_pct, x=labels, y=labels, text=text, texttemplate="%{text}",
    colorscale=[[0, "white"], [0.5, "#c7d2fe"], [1, COLORS["primary"]]],
    showscale=True, colorbar=dict(title="%")
))
fig.update_layout(title="Matriz de confusão — Classificação de risco (% por linha)",
                  xaxis_title="Previsto", yaxis_title="Real",
                  font=dict(family="DM Sans"), template="plotly_white",
                  width=600, height=550, yaxis=dict(autorange="reversed"))
fig.show()

## Resumo dos resultados

| Aspecto | Resultado |
|---------|----------|
| Melhor modelo | B_inmet_enriquecido (R²_log = 0.8878) |
| Ganho INMET | +0.6% R²_log (SINAN-only → enriquecido) |
| Série longa (B vs A) | +1.5% R²_log |
| Decay (C vs B) | Marginal (-0.1%) |
| Classificação | F1_macro funcional para 4 classes |

**Dashboard interativa:** [modelo-previsao-ffqc3ugltpbxxeaa4mygvw.streamlit.app](https://modelo-previsao-ffqc3ugltpbxxeaa4mygvw.streamlit.app)